In [1]:
%matplotlib widget
# Boilerplate import code for all libraries
# Changes to the precision require re-loading the kernel and need to be done before any op uses them.
import sphWarpCore_config as swc
from typing import Any
swc.configure(precision="float32", dim=Any) # precision: float16|half|float32|single|float64|double

import sphWarpCore as sph
from sphWarpCore.type_config import *
print(get_type_config()) # confirms active settings

# Initialize warp at this point
import warp as wp; wp.init()

import os
import torch
if torch.cuda.is_available(): # set the TORCH_CUDA_ARCH_LIST environment variable to the compute capability of the GPU for faster compiles
    os.environ['TORCH_CUDA_ARCH_LIST'] = f'{torch.cuda.get_device_properties(0).major}.{torch.cuda.get_device_properties(0).minor}'

import warnings
from tqdm import TqdmExperimentalWarning
warnings.filterwarnings("ignore", category=TqdmExperimentalWarning)
from tqdm.autonotebook import tqdm

# final import blocks that are generic
import matplotlib.pyplot as plt
from torch.profiler import profile, record_function, ProfilerActivity
import numpy as np
import math
import shlex    
import subprocess
import shutil

# custom SPH libraries
from integrators.integration import *
from sphWarpCore import *
from warpPlot import *

# This library
from warpSPH import *

# The case utilities that contain all the case setup functions for the various test cases
from warpSPH.caseUtils import *

{'scalar_t': <class 'warp._src.types.float32'>, 'dim_t': typing.Any}
Warp 1.15.0 initialized:
   CUDA Toolkit 12.9, Driver 13.3
   Devices:
     "cpu"      : "CPU"
     "cuda:0"   : "NVIDIA GeForce RTX 3090" (24 GiB, sm_86, mempool enabled)
   Kernel cache:
     /home/veldora/.cache/warp/1.15.0


In [25]:
directories = []

directories.append(f'export/semiPeriodic/')
directories.append(f'export/dambreak/')
directories.append(f'export/fullyPeriodic/')
directories.append(f'export/openChannel/')
directories.append(f'export/boundedRandom/')
directories.append(f'export/boundedRandom_wObstacle/')
directories.append(f'export/periodicRandom/')
directories.append(f'export/periodicRandom_wObstacle/')

# for now just take the newest subdirectory in each directory
simulationDirectories = []
for directory in directories:
    subdirs = [f.path for f in os.scandir(directory) if f.is_dir()]
    if len(subdirs) > 0:
        subdirs.sort(key=lambda x: os.path.getmtime(x), reverse=True)
        simulationDirectories.append(subdirs[0])
print(f"Found {len(simulationDirectories)} simulation directories: {simulationDirectories}")

directory = simulationDirectories[2]

trajectoryFile = f'{directory}/trajectory.h5'
configFile = f'{directory}/config.json'

Found 8 simulation directories: ['export/semiPeriodic/2026-07-29_00-55-21_256_4_2.0_6.0_obstacle_0.375_0_-1.5', 'export/dambreak/2026-07-28_23-08-00_256_4_2.0_4.0_obstacle_0.25_0_1', 'export/fullyPeriodic/2026-07-29_10-15-22_256_4_2.0_6.0_obstacle_0.375_0_-1.5', 'export/openChannel/2026-07-28_23-59-36_256_4_2.0_6.0_obstacle_0.25_0_-1.5', 'export/boundedRandom/2026-07-29_03-12-06_256_4_2.0_2.0_no_obstacle', 'export/boundedRandom_wObstacle/2026-07-29_04-08-25_256_4_2.0_2.0_obstacle_0.5_0_0', 'export/periodicRandom/2026-07-29_05-07-09_256_4_2.0_2.0_no_obstacle', 'export/periodicRandom_wObstacle/2026-07-29_05-38-13_256_4_2.0_2.0_obstacle_0.5_0_0']


In [26]:
import h5py
import json

loadedConfig = json.load(open(configFile, 'r'))

trajectory = h5py.File(trajectoryFile, 'r')
numStates = len(trajectory['states'])
print(f'Loaded trajectory with {numStates} states from {trajectoryFile}')

Loaded trajectory with 600 states from export/fullyPeriodic/2026-07-29_10-15-22_256_4_2.0_6.0_obstacle_0.375_0_-1.5/trajectory.h5


In [27]:
def load_state(stateIndex, file):
    frame_keys = list(file['states'].keys())
    frame_key = frame_keys[stateIndex] if 0 <= stateIndex < len(frame_keys) else frame_keys[0]

    state = file['states'][frame_key]
    positions = torch.tensor(state['positions'][:], dtype=torch.float32)
    velocities = torch.tensor(state['velocities'][:], dtype=torch.float32)
    densities = torch.tensor(state['densities'][:], dtype=torch.float32)
    masses = torch.tensor(file['initialState']['masses'][:], dtype=torch.float32)
    supports = torch.tensor(file['initialState']['supports'][:], dtype=torch.float32)
    kinds = torch.tensor(file['initialState']['kinds'][:], dtype=torch.int32)
    UIDs = torch.tensor(file['initialState']['UIDs'][:], dtype=torch.int64)

    return WeaklyCompressibleState(
        positions=positions,
        velocities=velocities,
        densities=densities,
        masses=masses,
        supports=supports,
        kinds=kinds,
        UIDs=UIDs,
        materials = torch.zeros_like(kinds, dtype=torch.int32),
        UIDcounter = UIDs.max().cpu().item() + 1
    ), state.attrs['time']


In [28]:
caseName = trajectory.attrs['caseName']
timeLimit = trajectory.attrs['timeLimit']
# dt = trajectory['states']['frame_00001'].attrs['time'] - trajectory['states']['frame_00000'].attrs['time']
nx = trajectory.attrs['nx']
n_h = trajectory.attrs['n_h']
L = trajectory.attrs['L']
W = trajectory.attrs['W']
obstacleType = trajectory.attrs['obstacleType']
aoa = trajectory.attrs['aoa']
obstacleActive = trajectory.attrs['obstacleActive']

dt = loadedConfig['config']['dt']
fixedSoundSpeed = loadedConfig['schemeConfig']['fixedSoundSpeed']

device = torch.device('cpu')
dtype = torch.float32

# dx = loadedConfig['config']['dx']
# band = trajectory.attrs['band']
dim = loadedConfig['config']['dim']
domain = buildDomainDescription(L, dim, True, device, dtype)
domain.min = torch.tensor(loadedConfig['config']['domain']['min'], device=device, dtype=dtype)
domain.max = torch.tensor(loadedConfig['config']['domain']['max'], device=device, dtype=dtype)

In [29]:
markerSize = 6
plotWidth = 28
plotHeight = 10

In [34]:
stateIndex = 0
state, time = load_state(stateIndex, trajectory)

caseText = f'{caseName}'
timeText = f't = {time:.4g}/{timeLimit:.4g} | dt = {dt:.4g}'
particleText = f'particles = {len(state.positions[state.kinds == 0])} fluid + {len(state.positions[state.kinds == 1])} boundary | nx = {nx} | n_h = {n_h}'
domainText = f'L = {L}, W = {W}'
obstacleText = f'obstacle: {obstacleType}, aoa: {aoa}' if obstacleActive else 'no obstacle'
stateText = f'v_max = {state.velocities.max().cpu().item():.4g} (c0 = {fixedSoundSpeed:.4g}), rho_max = {state.densities.max().cpu().item():.4g}, rho_min = {state.densities.min().cpu().item():.4g}'
timingText = f'iter time: {0.00:.3f} ms'

titleString = f'{caseText} | {timeText} | {particleText} | {domainText} | {obstacleText} | {stateText} | {timingText}'

from ipywidgets import widgets

def update_frame(frame_index):
    global state, time
    state, time = load_state(frame_index, trajectory)
    plotter.updateQuantities({
        # "A": state.velocities,
        "A": state.UIDs
    }, newParticleState=state)
    caseText = f'{caseName}'
    timeText = f't = {time:.4g}/{timeLimit:.4g} | dt = {dt:.4g}'
    particleText = f'particles = {len(state.positions[state.kinds == 0])} fluid + {len(state.positions[state.kinds == 1])} boundary | nx = {nx} | n_h = {n_h}'
    domainText = f'L = {L}, W = {W}'
    obstacleText = f'obstacle: {obstacleType}, aoa: {aoa}' if obstacleActive else 'no obstacle'
    stateText = f'v_max = {state.velocities.max().cpu().item():.4g} (c0 = {fixedSoundSpeed:.4g}), rho_max = {state.densities.max().cpu().item():.4g}, rho_min = {state.densities.min().cpu().item():.4g}'
    timingText = f'iter time: {0.00:.3f} ms'

    titleString = f'{caseText} | {timeText} | {particleText} | {domainText} | {obstacleText} | {stateText} | {timingText}'

    plotter.updateTitle(titleString)


markerSize = 12
velocityPlot = PlottingOptions(
            colorMap = UniformColorMap.viridis,
            markerSize = markerSize,
            midPoint = 0.0,
            quantityScaling = PlotScaling.Linear,
            mapping = Mapping.L2Norm,
            plotTitle = "Particle Velocity Magnitude",
            plotTitleGap = 0.08,
            boundaryVisualization = VisualizeOptions.Visualize,
            # gridVisualization = GridVisualization(
            #     resolution = 1024,
            #     streamLines = True,
            # ),

            # vMin=1e-10,
            vMin = 0.0,
            vMax = fixedSoundSpeed * 0.1,
        )
densityPlot = PlottingOptions(
            colorMap = DivergingColorMap.RdBu,
            flipColorMap=True,
            markerSize = markerSize,
            midPoint = 1.0,
            quantityScaling = PlotScaling.Linear,
            plotTitle = "Particle Density",
            # vMin = 0.95,
            # vMax = 1.05,
            plotTitleGap = 0.08,
            # gridVisualization = GridVisualization(
            #     resolution = 512,
            # ),
        )
UIDPlot = PlottingOptions(
            colorMap = CyclicColorMap.twilight,
            # flipColorMap=True,
            markerSize = markerSize,
            # midPoint = 1.0,
            quantityScaling = PlotScaling.Linear,
            plotTitle = f"Particle IDs ({caseName},  {len(state.positions[state.kinds == 0])} fluid + {len(state.positions[state.kinds == 1])} boundary particles)",
            boundaryVisualization = VisualizeOptions.Passive,
            # vMin = 0.95,
            # vMax = 1.05
            plotTitleGap = 0.08,
            # gridVisualization = GridVisualization(
            #     resolution = 512,
            # ),
        )


In [35]:

plotter = visualize(
    particleState = state,
    domain = domain,
    quantities = {
        # "A": state.velocities,
        "A": state.UIDs
    },
    plotOptions = {
        # "A": velocityPlot,
        "A": UIDPlot
    },
    figTitle = titleString,
    mosaic = 'A',
    figsize= (plotWidth, plotHeight),
    backend='vispy',
    # backend='pyVista',
    # backendOptions = {
    #     # In notebooks, use trame for reliable live updates.
    #     'jupyter_backend': 'trame',
    # }
)

plotter.updateTitle(titleString)


frame_slider = widgets.IntSlider(
    value=500,
    min=0,
    max=numStates-1,
    step=1,
    description='Frame:',
    continuous_update=True,
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='80%')
)

frame_slider.observe(lambda change: update_frame(change['new']), names='value')
update_frame(frame_slider.value)

export_button = widgets.Button(
    description='Export Current Frame',
    button_style='success',
    tooltip='Export the current frame as an image',
    icon='download'
)
export_button.on_click(lambda b: plotter.export(f'{caseName}_frame_{frame_slider.value:05d}.png'))


display(frame_slider)
display(export_button)


RFBOutputContext()

IntSlider(value=500, description='Frame:', layout=Layout(width='80%'), max=599, style=SliderStyle(description_…

Button(button_style='success', description='Export Current Frame', icon='download', style=ButtonStyle(), toolt…

In [120]:
uMax = torch.linalg.norm(state.velocities, dim=1).max().cpu().item()
dx = state.masses[state.kinds == 0].mean().cpu().item() ** (1.0 / dim)  # approximate particle spacing based on mass and density
cfl = uMax * dt / dx
print(f"Max velocity: {uMax:.4g}, dx: {dx:.4g}, dt: {dt:.4g}, CFL number: {cfl:.4g}")

targetCFL = 1.0
maxDt = targetCFL * dx / uMax
print(f"Target CFL: {targetCFL:.4g}, Max dt for target CFL: {maxDt:.4g}")
dtRatio = dt / maxDt
print(f"dt ratio: {dtRatio:.4g} (dt / maxDt for target CFL)")
print(f'Maximum Coarse Graining Factor: {1/dtRatio:.4g} (1 / dtRatio for target CFL)')

Max velocity: 2.292, dx: 0.008118, dt: 0.0002, CFL number: 0.05647
Target CFL: 1, Max dt for target CFL: 0.003541
dt ratio: 0.05647 (dt / maxDt for target CFL)
Maximum Coarse Graining Factor: 17.71 (1 / dtRatio for target CFL)
